# 06 · Personalisation with uncertainty: MAP, Laplace and conformal

**Goal:** adapt the twin to a new patient from a few days of data, attach honest error bars, and
check that they're honest.

**Time:** ~1.5 h · **You'll write:** `neg_log_posterior`, `jacobian_fd`, `ggn_hessian`,
`coverage`, `conformal_halfwidth`

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

from biotwin.data.dataset import Cohort, standard_sets, subset, LH, H
from biotwin.models.mechanistic import MinimalModel, CorrectionNet, PARAM_NAMES
from biotwin.uncertainty import ADAPT, ADAPT_IDX, PRIOR_SD, laplace_personalise, horizon_sigma, _full_theta
from biotwin.uncertainty import conformal_halfwidth as lib_conformal
from biotwin.train import predict
cohort = Cohort.load("data/cohort.npz")
S = standard_sets(cohort, seed=0)

## 1. Warm-up: a problem where we know the exact answer

Estimate a gain $a$ from $y_i = a\,x_i + \varepsilon_i$, with $\varepsilon_i \sim \mathcal N(0,\sigma^2)$
and prior $a \sim \mathcal N(m, s^2)$. The posterior is Gaussian, with

$$\text{precision} = \frac{\sum x_i^2}{\sigma^2} + \frac1{s^2}, \qquad \text{mean} = \frac{\sum x_i y_i/\sigma^2 + m/s^2}{\text{precision}}$$

That's the LMMSE / Kalman update you already know. The **negative log posterior** (up to a
constant) is

$$\mathcal L(\phi) = \tfrac12 \sum_i \left(\frac{y_i - \hat y_i(\phi)}{\sigma_i}\right)^2 + \tfrac12 \sum_j \left(\frac{\phi_j - \mu_j}{s_j}\right)^2$$

## Exercise 1 — `neg_log_posterior(pred, y, sigma, phi, mu, prior_sd)`


<details><summary><b>Hint</b></summary>

Two sums. `sigma` may be a scalar or a tensor that broadcasts with `pred`.

</details>

In [ ]:
def neg_log_posterior(pred, y, sigma, phi, mu, prior_sd):
    # TODO: return a scalar tensor
    raise NotImplementedError

def check(f):
    v = f(torch.tensor([1.0, 2.0]), torch.tensor([0.0, 0.0]), 2.0, torch.tensor([3.0]), torch.tensor([1.0]), torch.tensor([0.5]))
    close(v, 0.5 * (0.25 + 1.0) + 0.5 * 16.0, what="NLP")

neg_log_posterior = use(neg_log_posterior, sol.neg_log_posterior, check)

**Laplace approximation:** find the minimum $\hat\phi$ of $\mathcal L$ (the MAP), and approximate
the posterior as $\mathcal N(\hat\phi, \mathbf H^{-1})$, where $\mathbf H = \nabla^2 \mathcal L(\hat\phi)$.
For a linear model and a Gaussian prior this is **exact**. For a nonlinear model it's a local
Gaussian fit at the peak.

For least-squares losses there's a cheaper and safer curvature, **generalised Gauss–Newton**:

$$\mathbf H_{GGN} = \mathbf J^\top \Sigma^{-1} \mathbf J + \operatorname{diag}(1/s^2), \qquad \mathbf J_{ij} = \frac{\partial \hat y_i}{\partial \phi_j}$$

It drops the second-derivative-of-the-model term, so it's always positive semi-definite. It's
the Fisher information plus the prior precision.

## Exercise 2 — `jacobian_fd(fn, phi, eps)`

For `fn: [P] → any shape`, return the `[M, P]` Jacobian of the flattened output using **central**
differences, $(f(\phi + \varepsilon e_j) - f(\phi - \varepsilon e_j)) / 2\varepsilon$.


<details><summary><b>Hint</b></summary>

Loop over the P coordinates; `.reshape(-1)` each output difference; `torch.stack(cols, 1)`.

</details>

In [ ]:
def jacobian_fd(fn, phi, eps=1e-4):
    # TODO
    raise NotImplementedError

def check(f):
    fn = lambda p: torch.stack([torch.sin(p[0]) * p[1], p[0] ** 2, torch.exp(p[1] / 3)])
    phi = torch.tensor([0.3, -1.2], dtype=torch.float64)
    close(f(fn, phi), torch.autograd.functional.jacobian(fn, phi), tol=1e-6, what="Jacobian")

jacobian_fd = use(jacobian_fd, sol.jacobian_fd, check)

## Exercise 3 — `ggn_hessian(J, sigma, prior_sd)`

`J` is `[M, P]`, `sigma` is a scalar or `[M]` (one per residual), and `prior_sd` is `[P]`.


<details><summary><b>Hint</b></summary>

Whiten first: `Jw = J / sigma[:, None]`, then `Jw.T @ Jw + diag(1 / prior_sd**2)`.

</details>

In [ ]:
def ggn_hessian(J, sigma, prior_sd):
    # TODO
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    X = torch.randn(30, 3, generator=g, dtype=torch.float64); y = torch.randn(30, generator=g, dtype=torch.float64)
    sig = torch.rand(30, generator=g, dtype=torch.float64) + 0.5; mu = torch.zeros(3, dtype=torch.float64); sd = torch.tensor([1.0, 2.0, 0.5], dtype=torch.float64)
    # for a linear model, GGN equals the exact Hessian of the negative log posterior
    exact = torch.autograd.functional.hessian(lambda p: sol.neg_log_posterior(X @ p, y, sig, p, mu, sd), torch.zeros(3, dtype=torch.float64))
    close(f(X, sig, sd), exact, tol=1e-8, what="GGN")

ggn_hessian = use(ggn_hessian, sol.ggn_hessian, check)

Now a nonlinear 1-D case, where Laplace is only an approximation: a decay rate $k$ from
$y = e^{-k t} + \varepsilon$, with a **log-normal** prior (we fit $\phi = \log k$, as the twin does).
Compare the Laplace Gaussian with the exact posterior computed on a grid:

In [ ]:
torch.manual_seed(1)
t = torch.linspace(0, 3, 8, dtype=torch.float64); k_true, sig = 0.7, 0.08
y = torch.exp(-k_true * t) + sig * torch.randn(8, dtype=torch.float64)
model = lambda phi: torch.exp(-torch.exp(phi[0]) * t)
mu, sd = torch.tensor([0.0], dtype=torch.float64), torch.tensor([1.0], dtype=torch.float64)

grid = torch.linspace(-2, 1.5, 800, dtype=torch.float64)
nlp = torch.stack([neg_log_posterior(model(g[None]), y, sig, g[None], mu, sd) for g in grid])
post = torch.exp(-(nlp - nlp.min())); post /= torch.trapezoid(post, grid)
phi_map = grid[nlp.argmin()][None]
Hs = ggn_hessian(jacobian_fd(model, phi_map, 1e-5), torch.full((8,), sig, dtype=torch.float64), sd)
s_lap = Hs.inverse().sqrt().item()
lap = torch.exp(-0.5 * ((grid - phi_map) / s_lap) ** 2) / (s_lap * (2 * np.pi) ** 0.5)
plt.figure(figsize=(6, 3.2))
plt.plot(grid, post, label="exact (grid)"); plt.plot(grid, lap, "--", label="Laplace")
plt.axvline(np.log(k_true), c="k", lw=0.8, label="true log k"); plt.xlabel("φ = log k"); plt.legend(); plt.show()

Laplace works well when there's enough data for the posterior to be close to Gaussian. With
fewer points or more noise you'd see skew it can't represent. That's also why fitting **log**
parameters helps: the posterior looks more Gaussian in log space.

## 2. Personalising the twin

Same recipe on the real model. Freeze the network and adapt four physiological parameters
$\phi = \log(S_G, S_I, c_g, G_b)$ to one test patient, using that patient's days 0–4 (windows
whose targets don't overlap). The MAP step uses L-BFGS from the library. **You** compute the
covariance with your functions, and we compare it with the library's.

In [ ]:
hyb = MinimalModel(CorrectionNet(32)); hyb.load_state_dict(torch.load("results/checkpoints/hybrid.pt")); hyb.eval()
for p_ in hyb.parameters(): p_.requires_grad_(False)
pop = hyb.log_theta.detach().clone()
sig_h = horizon_sigma(predict(hyb, S["val"]), S["val"]["y"])         # per-horizon noise, from val patients

name = "adolescent#007"; pid = cohort.idx(name)
fw = subset(S["test_fit"], (S["test_fit"]["pid"] == pid) & (S["test_fit"]["t0"] % H == 0))
phi_map, cov_lib = laplace_personalise(hyb, fw, sig_h)

fn = lambda phi: hyb(fw, log_theta=_full_theta(pop, phi))
J = jacobian_fd(fn, phi_map, eps=1e-3)
sigma_vec = sig_h.expand(len(fw["y"]), -1).reshape(-1)
cov_mine = torch.linalg.inv(ggn_hessian(J, sigma_vec, PRIOR_SD))
print("your covariance matches the library's:", torch.allclose(cov_mine, cov_lib, rtol=1e-3, atol=1e-6))
for k, i, ph, s_ in zip(ADAPT, ADAPT_IDX, phi_map, cov_mine.diag().sqrt()):
    print(f"{k:3s} population {pop[i].exp():8.4g} -> MAP {ph.exp():8.4g}  (posterior sd in log space {s_:.2f}, prior {PRIOR_SD[ADAPT.index(k)]:.2f})")

Compare each posterior sd with the prior sd. Where they're about equal, the data taught us
nothing about that parameter (identifiability again). Where the posterior is much tighter, the
patient's data pins it down.

## 3. Are the error bars honest?

The only real test is **coverage** on held-out data: an 80 % interval must contain the truth
about 80 % of the time.

## Exercise 4 — `coverage(lo, hi, y)` (a float)


<details><summary><b>Hint</b></summary>

Fraction of entries with `lo <= y <= hi`.

</details>

In [ ]:
def coverage(lo, hi, y):
    # TODO
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    y = torch.randn(200000, generator=g)
    close(f(-1.2816 * torch.ones_like(y), 1.2816 * torch.ones_like(y), y), 0.8, tol=5e-3, what="coverage of N(0,1) in ±1.2816")

coverage = use(coverage, sol.coverage, check)

## Exercise 5 — split conformal

A distribution-free alternative. Take absolute residuals $|e_i|$ on a **calibration set** of
size $n$, per horizon. The half-width is their empirical quantile at level
$\min\!\big(1,\ \lceil (n+1)\,\ell \rceil / n\big)$. If calibration and test points are
exchangeable, this guarantees coverage $\ge \ell$.

`abs_resid` is `[n, H]`; return `[H]`.


<details><summary><b>Hint</b></summary>

`np.ceil`, then `torch.quantile(abs_resid, q, dim=0)`.

</details>

In [ ]:
def conformal_halfwidth(abs_resid, level):
    # TODO
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    p, y = torch.randn(300, 12, generator=g), torch.randn(300, 12, generator=g)
    close(f((p - y).abs(), 0.8), lib_conformal(p, y, 0.8), what="half-width")

conformal_halfwidth = use(conformal_halfwidth, sol.conformal_halfwidth, check)

In [ ]:
ev = subset(S["test_eval"], S["test_eval"]["pid"] == pid)
cal = subset(S["test_cal"], S["test_cal"]["pid"] == pid)
lt = _full_theta(pop, phi_map)
with torch.no_grad():
    pop_ev, map_ev, map_cal = hyb(ev), hyb(ev, log_theta=lt), hyb(cal, log_theta=lt)
sig_cal = horizon_sigma(map_cal, cal["y"])            # noise level from the calibration day (not the eval days)

# Laplace predictive: sample phi, run the twin, add noise
gen = torch.Generator().manual_seed(0)
L = torch.linalg.cholesky(cov_mine)
samples = []
with torch.no_grad():
    for _ in range(40):
        ph = phi_map + L @ torch.randn(4, generator=gen)
        samples.append(hyb(ev, log_theta=_full_theta(pop, ph)) + sig_cal * torch.randn(ev["y"].shape, generator=gen))
samples = torch.stack(samples)

z80 = 1.2816
hw = conformal_halfwidth((map_cal - cal["y"]).abs(), 0.8)
yk = ev["y"][:, -1]
print(f"{name}, evaluation days, 60-min horizon, 80% intervals:")
print(f"  population, σ from val    coverage {coverage(pop_ev[:, -1] - z80 * sig_h[-1], pop_ev[:, -1] + z80 * sig_h[-1], yk):.2f}  width {2 * z80 * sig_h[-1]:.0f}")
lo, hi = torch.quantile(samples[:, :, -1], 0.1, 0), torch.quantile(samples[:, :, -1], 0.9, 0)
print(f"  personalised, Laplace     coverage {coverage(lo, hi, yk):.2f}  width {(hi - lo).mean():.0f}")
print(f"  personalised, conformal   coverage {coverage(map_ev[:, -1] - hw[-1], map_ev[:, -1] + hw[-1], yk):.2f}  width {2 * hw[-1]:.0f}")
print(f"  RMSE 60 min: population {(pop_ev[:, -1] - yk).pow(2).mean().sqrt():.1f} -> personalised {(map_ev[:, -1] - yk).pow(2).mean().sqrt():.1f}")

For one patient and a few hundred correlated windows, coverage numbers move by a few
percent. The project-level results (all 6 patients) are in the main tutorial notebook, E3. The
pattern to look for: personalisation makes intervals **narrower** while keeping coverage near
nominal. The population model needs wide intervals because it doesn't know which patient it's
looking at.

## Quiz

Answer in your own words before opening the answers.

**1. Why is the Laplace approximation exact for a linear-Gaussian model?**

<details><summary>Answer</summary>

The log posterior is then exactly quadratic in φ (a Gaussian likelihood in a linear function, plus a Gaussian prior), so the posterior is exactly Gaussian, with mean at the MAP and covariance equal to the inverse Hessian. It's the LMMSE / Kalman update.

</details>

**2. Why use the Gauss–Newton Hessian instead of the exact Hessian of the nonlinear model?**

<details><summary>Answer</summary>

GGN = JᵀΣ⁻¹J + prior precision is always positive semi-definite, so its inverse is a valid covariance. The exact Hessian has an extra term with the model's second derivatives times the residuals, which can make it indefinite away from a perfect fit. GGN also only needs first derivatives (8 forward passes here).

</details>

**3. Overlapping windows reuse the same CGM samples many times. What does that do to the posterior, and what did we do about it?**

<details><summary>Answer</summary>

The likelihood counts the same evidence several times, so the posterior comes out overconfident (too narrow). We fit only on windows with non-overlapping targets (every 12th), and then check the intervals with coverage on held-out days instead of trusting the formula.

</details>

**4. Why estimate the observation noise σ on a separate calibration day instead of the fit days?**

<details><summary>Answer</summary>

Residuals on the data you fitted to are optimistically small (the parameters were tuned to them), so σ would be underestimated and the intervals would under-cover. A held-out day gives honest residuals. Conformal has the same logic: calibration must be separate from fitting.

</details>

**5. Conformal guarantees coverage ≥ ℓ. Why is that only approximately true here?**

<details><summary>Answer</summary>

The guarantee requires calibration and test points to be exchangeable. Windows from day 5 and from days 6–9 of a time series aren't strictly exchangeable (time-of-day effects, drift, autocorrelation), and the guarantee is marginal (on average over windows), not per window or per situation.

</details>